In [18]:
import pandas as pd

df_alliedvision = pd.read_csv("alliedvision.csv")
df_basler = pd.read_csv("basler.csv")
df_hikrobot = pd.read_csv("hikrobot.csv")
df_huaray = pd.read_csv("huaray.csv")
df_jai = pd.read_csv("jai.csv")
df_lucid = pd.read_csv("lucid.csv")
df_sentech = pd.read_csv("sentech.csv")
df_teledyne = pd.read_csv("teledyne.csv")

dfs = {
    "alliedvision": df_alliedvision,
    "basler": df_basler,
    "hikrobot": df_hikrobot,
    "huaray": df_huaray,
    "jai": df_jai,
    "lucid": df_lucid,
    "sentech": df_sentech,
    "teledyne": df_teledyne,
}


In [20]:
drop_3d = {
    "alliedvision": ("센서 유형", []),
    "basler":        ("카메라 카테고리", ["3D 카메라"]),
    "hikrobot":      ("camera_type", ["3D Binocular Structured Light", "3D Line Laser Camera", "3D RGB-D Smart Camera"]),
    "huaray":        ("camera_type", ["3D Camera"]),
    "jai":           (None, []),
    "lucid":         (None, []),
    "sentech":       (None, []),
    "teledyne":      ("camera_type", ["3D Camera"]),
}

drop_smart = {
    "alliedvision": ("센서 유형", []),
    "basler":        ("카메라 카테고리", []),
    "hikrobot":      ("camera_type", []),
    "huaray":        ("camera_type", ["AI Smart Camera"]),
    "jai":           (None, []),
    "lucid":         (None, []),
    "sentech":       (None, []),
    "teledyne":      ("camera_type", []),
}

# area, line만 있는 버전
dfs_area_line = {}
# 3d, smart 등 그외만 있는 버전
dfs_others = {}

for brand, df in dfs.items():
    col_3d, vals_3d = drop_3d[brand]
    col_smart, vals_smart = drop_smart[brand]

    exclude_vals = []
    exclude_col = col_3d or col_smart

    if col_3d and vals_3d:
        exclude_vals += vals_3d
    if col_smart and vals_smart:
        exclude_vals += vals_smart

    if exclude_col and exclude_vals:
        mask = df[exclude_col].isin(exclude_vals)
        df[~mask].to_csv(f"zz_{brand}_area_line.csv", index=False, encoding="utf-8-sig")
        df[mask].to_csv(f"zz_{brand}_others.csv", index=False, encoding="utf-8-sig")
    else:
        df.to_csv(f"zz_{brand}_area_line.csv", index=False, encoding="utf-8-sig")
        pd.DataFrame(columns=df.columns).to_csv(f"zz_{brand}_others.csv", index=False, encoding="utf-8-sig")

# area_line 파일로 column_compare_wide 만들기
dfs_area_line = {brand: pd.read_csv(f"zz_{brand}_area_line.csv", encoding="utf-8-sig") for brand in dfs.keys()}

dataset_tables = []
for df_name, df in dfs_area_line.items():
    rows = []
    for col in df.columns:
        sample_values = df[col].dropna().astype(str).unique()[:5]
        rows.append({
            f"{df_name}_컬럼명": col,
            f"{df_name}_샘플데이터": " | ".join(map(str, sample_values))
        })
    dataset_tables.append(pd.DataFrame(rows))

pd.concat(dataset_tables, axis=1).to_csv("column_compare_wide.csv", index=False, encoding="utf-8-sig")
print("저장 완료")

저장 완료


In [1]:
import pandas as pd

df_alliedvision = pd.read_csv("zz_alliedvision_area_line.csv")
df_basler = pd.read_csv("zz_basler_area_line.csv")
df_hikrobot = pd.read_csv("zz_hikrobot_area_line.csv")
df_huaray = pd.read_csv("zz_huaray_area_line.csv")
df_jai = pd.read_csv("zz_jai_area_line.csv")
df_lucid = pd.read_csv("zz_lucid_area_line.csv")
df_sentech = pd.read_csv("zz_sentech_area_line.csv")
df_teledyne = pd.read_csv("zz_teledyne_area_line.csv")

dfs = {
    "alliedvision": df_alliedvision,
    "basler": df_basler,
    "hikrobot": df_hikrobot,
    "huaray": df_huaray,
    "jai": df_jai,
    "lucid": df_lucid,
    "sentech": df_sentech,
    "teledyne": df_teledyne,
}

In [46]:

compare = pd.read_csv("column_compare_wide.csv", encoding="utf-8-sig")

keywords = ["스펙트럼", "spectrum", "Spectrum" "capability", "mount", "Mount" "인터페이스", "interface", "Interface", "mono", "Mono" ]
pattern = "|".join(keywords)

rows = []

for brand, df in dfs.items():
    col_name_col = f"{brand}_컬럼명"
    col_sample_col = f"{brand}_샘플데이터"

    mask = compare[col_name_col].astype(str).str.contains(pattern, case=False, na=False)
    matched_cols = compare[col_name_col][mask].dropna().unique()

    brand_rows = []
    for col in matched_cols:
        if col in df.columns:
            unique_vals = df[col].dropna().unique().tolist()
            brand_rows.append([brand, col] + unique_vals)

    if brand_rows:
        rows.extend(brand_rows)
        rows.append([])  # 브랜드 사이 빈 행

pd.DataFrame(rows).to_csv("check_data.csv", index=False, header=False, encoding="utf-8-sig")

In [74]:
import re
import os
import shutil
import pandas as pd


# ─────────────────────────────────────────────
# alliedvision 전처리
# ─────────────────────────────────────────────

if not os.path.exists("zz_alliedvision_area_line_1.csv"):
    shutil.copy("zz_alliedvision_area_line.csv", "zz_alliedvision_area_line_1.csv")
    print("원본 백업 완료 → zz_alliedvision_area_line_1.csv")
else:
    print("백업 파일 이미 존재 → 백업 건너뜀")

_df_av = pd.read_csv("zz_alliedvision_area_line.csv", encoding="utf-8-sig")

def _av_combine(row):
    parts = []
    for col in ["모노크롬 픽셀 형식", "베이어 픽셀 형식", "RGB 픽셀 형식", "YUV 픽셀 형식"]:
        v = str(row.get(col, "")).strip()
        if v and v != "nan":
            parts.append(v)
    return " | ".join(parts)

_df_av["_pixel_format_combined"] = _df_av.apply(_av_combine, axis=1)

_df_av.to_csv("zz_alliedvision_area_line.csv", index=False, encoding="utf-8-sig")
print("alliedvision combined 컬럼 추가 완료 → zz_alliedvision_area_line.csv")


# ─────────────────────────────────────────────
# teledyne 전처리
# ─────────────────────────────────────────────

if not os.path.exists("zz_teledyne_area_line_1.csv"):
    shutil.copy("zz_teledyne_area_line.csv", "zz_teledyne_area_line_1.csv")
    print("원본 백업 완료 → zz_teledyne_area_line_1.csv")
else:
    print("백업 파일 이미 존재 → 백업 건너뜀")

_df = pd.read_csv("zz_teledyne_area_line.csv", encoding="utf-8-sig")

def _c(col1, col2):
    a = _df[col1].fillna("").astype(str).str.strip()
    b = _df[col2].fillna("").astype(str).str.strip()
    return a.where(a != "", b)

# dimension: 크기 [W x H x D] → Dimensions [W x H x L]
_df["_dim_combined"]          = _c("크기 [W x H x D]", "Dimensions [W x H x L]")
# weight: 질량 → Mass
_df["_weight_combined"]       = _c("질량", "Mass")
# sensor_architecture: Sensor Technology → Sensor Type → 센서 유형
_st1 = _df["Sensor Technology"].fillna("").astype(str).str.strip()
_st2 = _df["Sensor Type"].fillna("").astype(str).str.strip()
_st3 = _df["센서 유형"].fillna("").astype(str).str.strip()
_df["_sensor_arch_combined"]  = _st1.where(_st1 != "", _st2).where(lambda x: x != "", _st3)

_df.to_csv("zz_teledyne_area_line.csv", index=False, encoding="utf-8-sig")
print("combined 컬럼 추가 완료 → zz_teledyne_area_line.csv")


# ─────────────────────────────────────────────
# pixel_format
# ─────────────────────────────────────────────

def _parse_pixel_format(v):
    """
    줄바꿈/쉼표/슬래시 구분 정리, 앞뒤 공백 제거
    'Mono 8,\nBayer GB 8/10...' → 'Mono8, BayerGB8/10...'
    """
    if not isinstance(v, str) or not v.strip():
        return ""
    # 줄바꿈 → 쉼표
    v = re.sub(r'\n', ', ', v)
    # 공백 여러 개 → 하나
    v = re.sub(r'\s+', ' ', v).strip()
    return v


def _alliedvision_pixel_format(mono, bayer, rgb, yuv):
    """alliedvision 4개 컬럼 합치기"""
    parts = []
    for val in [mono, bayer, rgb, yuv]:
        if isinstance(val, str) and val.strip():
            parts.append(val.strip())
    return " | ".join(parts) if parts else ""


# ─────────────────────────────────────────────
# sensor_architecture
# ─────────────────────────────────────────────

_ARCH_NORMALIZE = {
    "cmos":              "CMOS",
    "scmos":             "sCMOS",
    "ccd":               "CCD",
    "hybrid ccd/cmos":   "Hybrid CCD/CMOS",
    "ingaas":            "InGaAs",
    "linear ingaas photodiode array": "InGaAs",
    "hybrid cmosvisgaas and hgcdte (teledyne exclusive)": "VisGaAs/HgCdTe",
    "bsi":               "BSI CMOS",
    "fsi":               "FSI CMOS",
    "stacked bsi":       "Stacked BSI CMOS",
}

def _parse_sensor_arch(v):
    """
    'CMOS' → 'CMOS'
    'CMOS, global shutter' → 'CMOS'  (shutter 부분 제거)
    '1/4"CMOS' → 'CMOS'
    '3 x CMOS Vis/NIR/NIR' → 'CMOS'
    'Sony IMX183 CMOS (BSI Starvis)' → 'BSI CMOS'
    'sCMOS' → 'sCMOS'
    'InGaAs' → 'InGaAs'
    'Hybrid CCD/CMOS' → 'Hybrid CCD/CMOS'
    'Teledyne LACera CMOS' → 'CMOS'
    """
    if not isinstance(v, str) or not v.strip():
        return ""
    v_lower = v.strip().lower()

    # 정규화 맵 전체 매칭
    if v_lower in _ARCH_NORMALIZE:
        return _ARCH_NORMALIZE[v_lower]

    # BSI/FSI/Stacked 먼저 확인
    if re.search(r'stacked\s*bsi', v, re.I):
        return "Stacked BSI CMOS"
    if re.search(r'\bbsi\b', v, re.I):
        return "BSI CMOS"
    if re.search(r'\bfsi\b', v, re.I):
        return "FSI CMOS"

    # sCMOS
    if re.search(r'\bscmos\b', v, re.I):
        return "sCMOS"

    # InGaAs
    if re.search(r'\bingaas\b', v, re.I):
        return "InGaAs"

    # Hybrid CCD/CMOS
    if re.search(r'hybrid.*ccd.*cmos|hybrid.*cmos.*ccd', v, re.I):
        return "Hybrid CCD/CMOS"

    # CCD (CMOS보다 먼저)
    if re.search(r'\bccd\b', v, re.I) and not re.search(r'\bcmos\b', v, re.I):
        return "CCD"

    # CMOS
    if re.search(r'\bcmos\b', v, re.I):
        return "CMOS"

    return ""


# ─────────────────────────────────────────────
# weight_g
# ─────────────────────────────────────────────

def _parse_weight(v):
    """
    '70 g'                    → '70'
    '100 g'                   → '100'
    'Approx. 116 g (0.3 lb.)' → '116'
    'Approximately 210 g'     → '210'
    '2.3 kg'                  → '2300'
    '297 g'                   → '297'
    '116g'                    → '116'
    '30 g, (No lens mount 20 g)' → '30'
    '1420 g'                  → '1420'
    """
    if not isinstance(v, str) or not v.strip():
        return ""
    # kg → g 변환
    m_kg = re.search(r'([\d.]+)\s*kg\b', v, re.I)
    if m_kg:
        return str(int(float(m_kg.group(1)) * 1000))
    # g 첫번째 값만 추출
    m_g = re.search(r'([\d.]+)\s*g\b', v, re.I)
    if m_g:
        val = m_g.group(1)
        # 소수점이면 반올림
        return str(int(float(val))) if '.' in val else val
    return ""


# ─────────────────────────────────────────────
# dimension_w / dimension_h / dimension_l
# ─────────────────────────────────────────────

def _extract_3vals(v):
    """
    WxHxL 형태에서 세 숫자 추출 (mm 단위 제거, 인치 제거, 괄호 설명 제거)
    반환: (v1, v2, v3) 문자열 튜플
    """
    if not isinstance(v, str) or not v.strip():
        return ("", "", "")
    # 괄호 안 인치 표기 제거 (1.1" × 1.1" × 1.7")
    v = re.sub(r'\([^)]*"[^)]*\)', '', v)
    # 괄호 안 설명 제거 (not including...)
    v = re.sub(r'\([^)]*[a-zA-Z가-힣]{3,}[^)]*\)', '', v)
    # * 제거 (28 x 36 x 32.8* mm)
    v = re.sub(r'\*', '', v)
    # (W), (H), (D), (.W) 등 레이블 제거
    v = re.sub(r'\s*\(\.*[WHDLwhd]\.*\)\s*', ' ', v)
    # mm 제거
    v = re.sub(r'\s*mm\b', '', v, flags=re.I)
    # 숫자 3개 추출
    nums = re.findall(r'[\d.]+', v)
    if len(nums) >= 3:
        return (nums[0], nums[1], nums[2])
    return ("", "", "")


# 브랜드별 순서 매핑 → (W, H, L) 인덱스
# alliedvision: L x W x H → L=0, W=1, H=2
# basler:       L x W x H → L=0, W=1, H=2
# hikrobot:     W x H x L (=D) → W=0, H=1, L=2  (Depth = Length)
# huaray:       W x H x L → W=0, H=1, L=2
# jai:          H x W x L → H=0, W=1, L=2
# lucid:        W x H x L → W=0, H=1, L=2 (대부분), 일부 불규칙
# sentech:      W x H x D(=L) → W=0, H=1, L=2
# teledyne:     W x H x D(=L) → W=0, H=1, L=2

def _dim(v, order, idx):
    """
    order: (w_idx, h_idx, l_idx) — 원본 숫자 순서에서 W/H/L의 위치
    idx: 0=W, 1=H, 2=L
    """
    nums = _extract_3vals(v)
    if not any(nums):
        return ""
    pos = order[idx]
    return nums[pos] if pos < len(nums) else ""

# 각 브랜드 order: (w_pos, h_pos, l_pos)
_ORDER = {
    "alliedvision": (1, 2, 0),   # L×W×H → W=1, H=2, L=0
    "basler":       (1, 2, 0),   # L×W×H → W=1, H=2, L=0
    "hikrobot":     (0, 1, 2),   # W×H×L
    "huaray":       (0, 1, 2),   # W×H×L
    "jai":          (1, 0, 2),   # H×W×L → W=1, H=0, L=2
    "lucid":        (0, 1, 2),   # W×H×L
    "sentech":      (0, 1, 2),   # W×H×D(=L)
    "teledyne":     (0, 1, 2),   # W×H×D(=L)
}


def _make_dim_fn(brand, idx):
    order = _ORDER[brand]
    return lambda v: _dim(v, order, idx)


# ─────────────────────────────────────────────
# column_rules
# ─────────────────────────────────────────────

column_rules = {

    "alliedvision": [
        {"source_col": "_pixel_format_combined",    "target_col": "pixel_format",        "value_map": _parse_pixel_format},
        {"source_col": "센서 아키텍처 (소재)",       "target_col": "sensor_architecture", "value_map": _parse_sensor_arch},
        {"source_col": "무게",                       "target_col": "weight_g",            "value_map": _parse_weight},
        {"source_col": "본체 치수 (L x W x H mm)",  "target_col": "dimension_w",         "value_map": _make_dim_fn("alliedvision", 0)},
        {"source_col": "본체 치수 (L x W x H mm)",  "target_col": "dimension_h",         "value_map": _make_dim_fn("alliedvision", 1)},
        {"source_col": "본체 치수 (L x W x H mm)",  "target_col": "dimension_l",         "value_map": _make_dim_fn("alliedvision", 2)},
    ],

    "basler": [
        {"source_col": None,                   "target_col": "pixel_format",        "value_map": None},
        {"source_col": "센서 타입",            "target_col": "sensor_architecture", "value_map": _parse_sensor_arch},
        {"source_col": "무게 (Weight)",        "target_col": "weight_g",            "value_map": _parse_weight},
        {"source_col": "하우징 \u200d사이즈 (L x W x H)", "target_col": "dimension_w", "value_map": _make_dim_fn("basler", 0)},
        {"source_col": "하우징 \u200d사이즈 (L x W x H)", "target_col": "dimension_h", "value_map": _make_dim_fn("basler", 1)},
        {"source_col": "하우징 \u200d사이즈 (L x W x H)", "target_col": "dimension_l", "value_map": _make_dim_fn("basler", 2)},
    ],

    "hikrobot": [
        {"source_col": "Pixel format",  "target_col": "pixel_format",        "value_map": _parse_pixel_format},
        {"source_col": "Sensor type",   "target_col": "sensor_architecture", "value_map": _parse_sensor_arch},
        {"source_col": "Weight",        "target_col": "weight_g",            "value_map": _parse_weight},
        {"source_col": "Dimension",     "target_col": "dimension_w",         "value_map": _make_dim_fn("hikrobot", 0)},
        {"source_col": "Dimension",     "target_col": "dimension_h",         "value_map": _make_dim_fn("hikrobot", 1)},
        {"source_col": "Dimension",     "target_col": "dimension_l",         "value_map": _make_dim_fn("hikrobot", 2)},
    ],

    "huaray": [
        {"source_col": "Image Format",      "target_col": "pixel_format",        "value_map": _parse_pixel_format},
        {"source_col": "Image Sensor",      "target_col": "sensor_architecture", "value_map": _parse_sensor_arch},
        {"source_col": "Net Weight",        "target_col": "weight_g",            "value_map": _parse_weight},
        {"source_col": "Product Dimensions","target_col": "dimension_w",         "value_map": _make_dim_fn("huaray", 0)},
        {"source_col": "Product Dimensions","target_col": "dimension_h",         "value_map": _make_dim_fn("huaray", 1)},
        {"source_col": "Product Dimensions","target_col": "dimension_l",         "value_map": _make_dim_fn("huaray", 2)},
    ],

    "jai": [
        {"source_col": None,                    "target_col": "pixel_format",        "value_map": None},
        {"source_col": "Sensors",               "target_col": "sensor_architecture", "value_map": _parse_sensor_arch},
        {"source_col": "Weight",                "target_col": "weight_g",            "value_map": _parse_weight},
        {"source_col": "Camera Dimensions HxWxL","target_col": "dimension_w",        "value_map": _make_dim_fn("jai", 0)},
        {"source_col": "Camera Dimensions HxWxL","target_col": "dimension_h",        "value_map": _make_dim_fn("jai", 1)},
        {"source_col": "Camera Dimensions HxWxL","target_col": "dimension_l",        "value_map": _make_dim_fn("jai", 2)},
    ],

    "lucid": [
        {"source_col": "픽셀 형식",  "target_col": "pixel_format",        "value_map": _parse_pixel_format},
        {"source_col": "센서",       "target_col": "sensor_architecture", "value_map": _parse_sensor_arch},
        {"source_col": "무게",       "target_col": "weight_g",            "value_map": _parse_weight},
        {"source_col": "제품 사이즈","target_col": "dimension_w",         "value_map": _make_dim_fn("lucid", 0)},
        {"source_col": "제품 사이즈","target_col": "dimension_h",         "value_map": _make_dim_fn("lucid", 1)},
        {"source_col": "제품 사이즈","target_col": "dimension_l",         "value_map": _make_dim_fn("lucid", 2)},
    ],

    "sentech": [
        {"source_col": "Image Pixel Format","target_col": "pixel_format",        "value_map": _parse_pixel_format},
        {"source_col": None,                "target_col": "sensor_architecture", "value_map": None},
        {"source_col": "Weight",            "target_col": "weight_g",            "value_map": _parse_weight},
        {"source_col": "Dimensions",        "target_col": "dimension_w",         "value_map": _make_dim_fn("sentech", 0)},
        {"source_col": "Dimensions",        "target_col": "dimension_h",         "value_map": _make_dim_fn("sentech", 1)},
        {"source_col": "Dimensions",        "target_col": "dimension_l",         "value_map": _make_dim_fn("sentech", 2)},
    ],

    "teledyne": [
        {"source_col": None,                    "target_col": "pixel_format",        "value_map": None},
        {"source_col": "_sensor_arch_combined", "target_col": "sensor_architecture", "value_map": _parse_sensor_arch},
        {"source_col": "_weight_combined",      "target_col": "weight_g",            "value_map": _parse_weight},
        {"source_col": "_dim_combined",         "target_col": "dimension_w",         "value_map": _make_dim_fn("teledyne", 0)},
        {"source_col": "_dim_combined",         "target_col": "dimension_h",         "value_map": _make_dim_fn("teledyne", 1)},
        {"source_col": "_dim_combined",         "target_col": "dimension_l",         "value_map": _make_dim_fn("teledyne", 2)},
    ],
}

백업 파일 이미 존재 → 백업 건너뜀
alliedvision combined 컬럼 추가 완료 → zz_alliedvision_area_line.csv
백업 파일 이미 존재 → 백업 건너뜀
combined 컬럼 추가 완료 → zz_teledyne_area_line.csv


In [75]:
before_version = "v6"
new_version = "v7"

for brand, rules in column_rules.items():
    df_before = pd.read_csv(f"aa_{brand}_{before_version}.csv", encoding="utf-8-sig")
    df_origin = pd.read_csv(f"zz_{brand}_area_line.csv", encoding="utf-8-sig")

    for rule in rules:
        if rule["source_col"] and rule["source_col"] in df_origin.columns:
            if rule.get("value_map"):
                df_before[rule["target_col"]] = df_origin[rule["source_col"]].map(rule["value_map"])
            else:
                df_before[rule["target_col"]] = df_origin[rule["source_col"]].values
        else:
            df_before[rule["target_col"]] = ""

        for new_col, extra in rule.get("extra_cols", {}).items():
            if isinstance(extra, dict) and "source_col" in extra:
                src = extra["source_col"]
                df_before[new_col] = df_origin[src].map(extra["value_map"])
            else:
                df_before[new_col] = df_origin[rule["source_col"]].map(extra)

    df_before.to_csv(f"aa_{brand}_{new_version}.csv", index=False, encoding="utf-8-sig")

print("저장 완료")

저장 완료


In [33]:
for brand in column_rules.keys():
    df_origin = pd.read_csv(f"zz_{brand}_area_line.csv", encoding="utf-8-sig")
    print(brand, "model_name" in df_origin.columns)

alliedvision False
basler True
hikrobot True
huaray True
jai False
lucid False
sentech True
teledyne True


In [62]:
## 제거

compare = pd.read_csv("column_compare_wide_1.csv", encoding="utf-8-sig")

for brand, rules in column_rules.items():
    cols_to_clear = []

    for rule in rules:
        if rule.get("source_col"):
            cols_to_clear.append(rule["source_col"])

        for new_col, extra in rule.get("extra_cols", {}).items():
            if isinstance(extra, dict) and "source_col" in extra:
                cols_to_clear.append(extra["source_col"])

    mask = compare[f"{brand}_컬럼명"].isin(cols_to_clear)
    compare.loc[mask, f"{brand}_컬럼명"] = ""
    compare.loc[mask, f"{brand}_샘플데이터"] = ""

compare.to_csv("column_compare_wide_1.csv", index=False, encoding="utf-8-sig")

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc4 in position 13: invalid continuation byte

In [58]:
## 파싱
import re

def parse_alliedvision_sensor(val):
    val = str(val) if pd.notna(val) else ""
    
    sensor_name = ""
    multi_cmos = ""

    if not val or val == "nan":
        return val, sensor_name, multi_cmos

    parts = [p.strip() for p in val.split("|")]
    
    # FPA 320 × 256 같은 경우 sensor_name 비우기
    if re.search(r'\d+\s*[×x]\s*\d+', parts[0]):
        sensor_name = ""
    # 3-CMOS 처리
    elif re.match(r'3-CMOS\s+(\S+)', parts[0]):
        match = re.match(r'3-CMOS\s+(\S+)', parts[0])
        sensor_name = match.group(1)
        multi_cmos = "3CMOS"
    else:
        sensor_name = parts[0]

    return val, sensor_name, multi_cmos

df = pd.read_csv("aa_alliedvision_v2.csv", encoding="utf-8-sig")
df_origin = pd.read_csv("zz_alliedvision_area_line.csv", encoding="utf-8-sig")

parsed = df_origin["센서 모델"].apply(parse_alliedvision_sensor)
df["sensor_name"] = parsed.apply(lambda x: x[1])
df["sensor_raw"] = parsed.apply(lambda x: x[0])
df["multi_cmos"] = parsed.apply(lambda x: x[2])

df.to_csv("aa_alliedvision_v3_1.csv", index=False, encoding="utf-8-sig")

In [ ]:
## 카메라 타입 체크

import re

keywords = ["area", "line", "3d", "3D"]
pattern = "|".join(keywords)

rows = []

for brand, df in dfs.items():
    brand_rows = []
    for col in df.columns:
        unique_vals = df[col].dropna().astype(str).unique().tolist()
        joined = " | ".join(unique_vals)
        if re.search(pattern, joined, re.IGNORECASE):
            brand_rows.append([brand, col] + unique_vals)
    
    if brand_rows:
        rows.extend(brand_rows)
        rows.append([])

pd.DataFrame(rows).to_csv("check_camera_type.csv", index=False, header=False, encoding="utf-8-sig")